# Partial elastic net — penalise the exogenous predictors only, δ=0.999, 6-year window

Throw in **all** exogenous predictors (GVZ, VIX, OVX, SPX RV, Crude RV, macro-release
dummy) and shrink **only those** with elastic net, leaving the HAR autoregressive core
(`x_d, x_w, x_m`) unpenalised. Does honest out-of-sample QLIKE improve over the simple
OLS models — `log HAR + GVZ` and `log HAR + GVZ + macro`?

**λ (alpha)** is chosen per forecast day by the same **walk-forward CV** rule as
`HAR_RegressionElasticnet3D.ipynb` (3 expanding folds, `l1_ratio=0.1`, QLIKE-in-levels
selection, unweighted Duan smearing). Penalising only the exog block is done by
**Frisch-Waugh-Lovell partialling**: residualise `y` and the exog block on the HAR core,
elastic-net the residuals, then recover the HAR coefficients by OLS.

| Model | Regressors | Regularised? |
|-------|------------|--------------|
| log HAR + GVZ              | HAR + log_GVZ                 | no (OLS) |
| log HAR + GVZ + macro      | HAR + log_GVZ + macro         | no (OLS) |
| log HAR + all-exog (OLS)   | HAR + all 6 exog              | no (OLS, kitchen-sink) |
| log HAR + all-exog (partial EN) | HAR + all 6 exog        | exog only (elastic net) |

Results saved to `partial_enet_comparison.csv`.

In [1]:
# ===========================================================================
# Cell 1 — Imports & data (+ fixed test parameters and EN grid)
# ===========================================================================
import numpy as np
import pandas as pd
from sklearn.linear_model import ElasticNet, enet_path
from sklearn.preprocessing import StandardScaler

data = pd.read_parquet("merged_RV_GVZ_with_macro_event.parquet")  # RV_gold, RV_ES, RV_crude, GVZ_close, macro_event
iv   = pd.read_parquet("cross_asset_iv.parquet")                 # VIX_close, OVX_close
rv = data["RV_gold"].astype(float)

TRADING_DAYS = 252
EPS = 1e-6                                  # QLIKE forecast floor
WINDOW = int(round(6.0 * TRADING_DAYS))     # 1512 trading days (6 years)
DELTA  = 0.999                              # geometric recency-decay factor

# Elastic-net selection grid (mirrors HAR_RegressionElasticnet3D.ipynb)
L1_RATIO = 0.1                              # near-ridge L1/L2 mix, fixed
ALPHAS   = np.logspace(-4, 1, 25)          # penalty grid
WF_EDGES = [0.40, 0.60, 0.80, 1.00]        # 3 expanding walk-forward folds

print(f"RV_gold: {len(rv)} obs, {rv.index.min().date()} .. {rv.index.max().date()}")
print(f"data cols: {list(data.columns)} | iv cols: {list(iv.columns)}")
print(f"WINDOW = {WINDOW} days (~6y),  DELTA = {DELTA}")
print(f"EN grid: {len(ALPHAS)} alphas in [{ALPHAS.min():.0e},{ALPHAS.max():.0e}], "
      f"l1_ratio={L1_RATIO}, walk-forward edges={WF_EDGES}")

RV_gold: 4015 obs, 2010-06-08 .. 2026-05-29
data cols: ['RV_gold', 'RV_crude', 'RV_ES', 'GVZ_close', 'macro_event'] | iv cols: ['VIX_close', 'OVX_close']
WINDOW = 1512 days (~6y),  DELTA = 0.999
EN grid: 25 alphas in [1e-04,1e+01], l1_ratio=0.1, walk-forward edges=[0.4, 0.6, 0.8, 1.0]


In [2]:
# ===========================================================================
# Cell 2 — Build design tables
# ===========================================================================
# Log-HAR components on x = log(RV_gold); exogenous day-t terms logged (known at close,
# no look-ahead). macro is the RAW 0/1 release dummy shifted to t+1 (scheduled, known at t).
for col in ["RV_gold", "GVZ_close", "RV_ES", "RV_crude"]:
    assert (data[col] > 0).all(), f"{col} has non-positive values; log undefined"
for col in ["VIX_close", "OVX_close"]:
    assert (iv[col] > 0).all(), f"{col} has non-positive values; log undefined"

x = np.log(rv)

def build_log_design(extra_cols):
    """Base log-HAR design + optional exogenous day-t columns (dict name->series)."""
    df = pd.DataFrame(index=rv.index)
    df["x_d"] = x
    df["x_w"] = x.rolling(5).mean()
    df["x_m"] = x.rolling(22).mean()
    for name, series in extra_cols.items():
        df[name] = series.reindex(rv.index)
    df["y_log"]   = x.shift(-1)        # log(RV_{t+1})
    df["y_level"] = rv.shift(-1)       # RV_{t+1} in levels, for QLIKE
    return df.dropna()

log_gvz   = np.log(data["GVZ_close"])
log_spx   = np.log(data["RV_ES"])
log_crude = np.log(data["RV_crude"])
log_vix   = np.log(iv["VIX_close"])
log_ovx   = np.log(iv["OVX_close"])
macro     = data["macro_event"].shift(-1).astype(float)   # day-(t+1) scheduled-release dummy

EXOG = ["log_GVZ", "log_VIX", "log_OVX", "log_RV_ES", "log_RV_crude", "macro"]
HAR  = ["x_d", "x_w", "x_m"]
all_exog = {"log_GVZ": log_gvz, "log_VIX": log_vix, "log_OVX": log_ovx,
            "log_RV_ES": log_spx, "log_RV_crude": log_crude, "macro": macro}

d_gvz       = build_log_design({"log_GVZ": log_gvz})                       # benchmark 1
d_gvz_macro = build_log_design({"log_GVZ": log_gvz, "macro": macro})       # benchmark 2
d_all       = build_log_design(all_exog)                                  # HAR + all 6 exog

for name, df in [("d_gvz", d_gvz), ("d_gvz_macro", d_gvz_macro), ("d_all", d_all)]:
    assert df.notna().all().all(), f"unexpected NaNs in {name}"
    assert d_gvz.index.equals(df.index), f"{name} index misaligned with d_gvz"
assert set(np.unique(d_all["macro"].to_numpy())) <= {0.0, 1.0}, "macro not 0/1"

print("d_gvz       cols:", list(d_gvz.columns),       f"({len(d_gvz)} rows)")
print("d_gvz_macro cols:", list(d_gvz_macro.columns), f"({len(d_gvz_macro)} rows)")
print("d_all       cols:", list(d_all.columns),       f"({len(d_all)} rows)")
d_all.head()

d_gvz       cols: ['x_d', 'x_w', 'x_m', 'log_GVZ', 'y_log', 'y_level'] (3993 rows)
d_gvz_macro cols: ['x_d', 'x_w', 'x_m', 'log_GVZ', 'macro', 'y_log', 'y_level'] (3993 rows)
d_all       cols: ['x_d', 'x_w', 'x_m', 'log_GVZ', 'log_VIX', 'log_OVX', 'log_RV_ES', 'log_RV_crude', 'macro', 'y_log', 'y_level'] (3993 rows)


,x_d,x_w,x_m,log_GVZ,log_VIX,log_OVX,log_RV_ES,log_RV_crude,macro,y_log,y_level
Date,,,,,,,,,,,
2010-07-08,2.702348,2.887899,2.773932,3.008648,3.246880,3.577389,2.948605,3.307625,0.0,2.723668,15.236102
2010-07-09,2.723668,2.800490,2.762134,2.979603,3.218076,3.542986,2.685524,3.119290,0.0,2.691347,14.751526
2010-07-12,2.691347,2.766723,2.755241,2.918311,3.195812,3.502550,2.779032,3.331252,0.0,2.593328,13.374209
2010-07-13,2.593328,2.694457,2.744754,2.943913,3.201119,3.492560,2.925677,3.260566,0.0,2.754495,15.713097
2010-07-14,2.754495,2.693037,2.755626,2.903617,3.214466,3.501344,2.912753,3.341328,1.0,2.722249,15.214503


In [3]:
# ===========================================================================
# Cell 3 — Helpers: QLIKE, recency weights, OLS loss series, partial-EN loss series
# ===========================================================================
START_DATE = d_gvz.index[WINDOW]   # 6y warm-up, common to all models


def _recency_weights(n, delta):
    """Geometric recency weights; newest row weight 1, age `k` older -> delta**k, mean-1."""
    if delta >= 1.0:
        return np.ones(n)
    ages = np.arange(n)[::-1]
    w = delta ** ages
    return w * (n / w.sum())

def _qlike(actual, forecast, eps=EPS):
    f = np.maximum(forecast, eps)
    r = actual / f
    return r - np.log(r) - 1.0, int((forecast <= eps).sum())


# --- Batched recency-weighted OLS per-day QLIKE loss series (for the OLS models) -----
# (copied verbatim from HAR_MCS_without_implied_vol.ipynb Cell 3)
def rolling_log_ols_loss_series(design, feat_cols, window, start_date=None, delta=1.0):
    if start_date is None:
        start_date = START_DATE
    X = np.column_stack([np.ones(len(design)), design[feat_cols].to_numpy()])
    yl  = design["y_log"].to_numpy()
    lvl = design["y_level"].to_numpy()
    idx = design.index
    N, p = X.shape
    t_all = np.arange(window, N)
    t_oos = t_all[idx[window:] >= start_date]
    starts = t_oos - window
    Xwins = np.lib.stride_tricks.sliding_window_view(X, window, axis=0)[starts].transpose(0, 2, 1)
    ywins = np.lib.stride_tricks.sliding_window_view(yl, window)[starts]
    w  = _recency_weights(window, delta); sw = np.sqrt(w)
    Xs = Xwins * sw[None, :, None]; ys = ywins * sw[None, :]
    A = np.einsum("nwi,nwj->nij", Xs, Xs)
    b = np.einsum("nwi,nw->ni", Xs, ys)
    beta = np.linalg.solve(A, b)
    fitted = np.einsum("nwp,np->nw", Xwins, beta)
    smear = np.einsum("nw,w->n", np.exp(ywins - fitted), w) / w.sum()
    x_pred = np.einsum("np,np->n", X[t_oos], beta)
    fc = np.exp(x_pred) * smear
    ac = lvl[t_oos]
    q, _ = _qlike(ac, fc)
    return pd.Series(q, index=idx[t_oos], name="qlike")


# --- Partial elastic net: penalise EXOG block only, HAR core unpenalised (FWL) ------
# Per forecast day t over window [t-W:t]:
#   A = [1, HAR core] (unpenalised), B = standardised exog (penalised), w = recency wts.
#   alpha chosen by walk-forward CV (QLIKE-in-levels) on FWL-residualised data; refit at
#   alpha* on the full window; forecast with an unweighted Duan smear. Returns per-day
#   QLIKE Series plus arrays of the selected alpha and the #non-zero exog coefs per day.
def _wproj_resid(A, Z, w):
    """Weighted residual of Z after projecting onto columns of A (W=diag(w))."""
    AtW = A.T * w                                   # (p, m)
    coef = np.linalg.solve(AtW @ A, AtW @ Z)        # (p, ...) weighted OLS of Z on A
    return Z - A @ coef, coef

def rolling_log_partial_enet_wfcv_loss_series(design, har_cols, exog_cols, window,
                                              start_date=None, delta=1.0,
                                              l1_ratio=L1_RATIO, alphas=ALPHAS,
                                              wf_edges=WF_EDGES):
    if start_date is None:
        start_date = START_DATE
    Ah  = np.column_stack([np.ones(len(design)), design[har_cols].to_numpy()])  # [1, HAR]
    Bf  = design[exog_cols].to_numpy()
    yl  = design["y_log"].to_numpy()
    lvl = design["y_level"].to_numpy()
    idx = design.index
    w   = _recency_weights(window, delta)
    edges = [int(round(f * window)) for f in wf_edges]
    folds = [(0, edges[k], edges[k + 1]) for k in range(len(edges) - 1)]
    losses, dates, asel, nnz = [], [], [], []

    for t in range(window, len(design)):
        if idx[t] < start_date:
            continue
        Aw = Ah[t - window:t]
        yw = yl[t - window:t]
        lw = lvl[t - window:t]
        sc = StandardScaler().fit(Bf[t - window:t])      # X-only scaler, full window
        Bw = sc.transform(Bf[t - window:t])

        # ---- walk-forward CV: average QLIKE over folds per alpha ----
        qsum = np.zeros(len(alphas))
        for tr0, v0, v1 in folds:
            A_tr, B_tr, y_tr = Aw[tr0:v0], Bw[tr0:v0], yw[tr0:v0]
            A_vl, B_vl, l_vl = Aw[v0:v1], Bw[v0:v1], lw[v0:v1]
            w_tr = w[tr0:v0] * ((v0 - tr0) / w[tr0:v0].sum())   # mean-1 on this block
            sw_tr = np.sqrt(w_tr)
            # FWL: residualise y and B on A (weighted), on the train block
            ry, _   = _wproj_resid(A_tr, y_tr, w_tr)
            RB, _   = _wproj_resid(A_tr, B_tr, w_tr)
            _, coefs, _ = enet_path(RB * sw_tr[:, None], ry * sw_tr,
                                    l1_ratio=l1_ratio, alphas=alphas, max_iter=5000)
            # recover HAR coefs per alpha: b1 = wOLS(A_tr, y_tr - B_tr b2)
            AtW = A_tr.T * w_tr
            resid_y = y_tr[:, None] - B_tr @ coefs               # (m_tr, n_a)
            b1 = np.linalg.solve(AtW @ A_tr, AtW @ resid_y)      # (p, n_a)
            tr_fit  = A_tr @ b1 + B_tr @ coefs                   # (m_tr, n_a)
            smear   = np.mean(np.exp(y_tr[:, None] - tr_fit), axis=0)   # (n_a,) unweighted Duan
            val_log = A_vl @ b1 + B_vl @ coefs                   # (m_val, n_a)
            val_fc  = np.maximum(np.exp(val_log) * smear, EPS)
            r = l_vl[:, None] / val_fc
            qsum += np.mean(r - np.log(r) - 1.0, axis=0)
        a_star = float(alphas[int(np.argmin(qsum))])

        # ---- refit at alpha* on the full window (recency-weighted) ----
        ry_f, _ = _wproj_resid(Aw, yw, w)
        RB_f, _ = _wproj_resid(Aw, Bw, w)
        m = ElasticNet(alpha=a_star, l1_ratio=l1_ratio, fit_intercept=False,
                       max_iter=5000).fit(RB_f, ry_f, sample_weight=w)
        b2 = m.coef_
        AtW = Aw.T * w
        b1 = np.linalg.solve(AtW @ Aw, AtW @ (yw - Bw @ b2))
        fitted = Aw @ b1 + Bw @ b2
        smear = np.mean(np.exp(yw - fitted))                     # unweighted Duan
        x_b = sc.transform(Bf[t:t + 1])[0]
        log_pred = Ah[t] @ b1 + x_b @ b2
        fc = np.exp(log_pred) * smear
        q, _ = _qlike(np.array([lvl[t]]), np.array([fc]))
        losses.append(q[0]); dates.append(idx[t])
        asel.append(a_star); nnz.append(int(np.sum(b2 != 0.0)))

    return (pd.Series(losses, index=pd.DatetimeIndex(dates), name="qlike"),
            np.array(asel), np.array(nnz))


print(f"Common OOS start: {START_DATE.date()}  "
      f"({int((d_gvz.index >= START_DATE).sum())} forecast days)")

Common OOS start: 2016-07-11  (2481 forecast days)


In [4]:
# ===========================================================================
# Cell 4 — Compare 4 models with an MCS test, save partial_enet_comparison.csv
# ===========================================================================
from arch.bootstrap import MCS

loss_cols = {}
loss_cols["log HAR + GVZ"]         = rolling_log_ols_loss_series(d_gvz,       HAR + ["log_GVZ"], WINDOW, START_DATE, DELTA)
loss_cols["log HAR + GVZ + macro"] = rolling_log_ols_loss_series(d_gvz_macro, HAR + ["log_GVZ", "macro"], WINDOW, START_DATE, DELTA)
loss_cols["log HAR + all-exog (OLS)"] = rolling_log_ols_loss_series(d_all, HAR + EXOG, WINDOW, START_DATE, DELTA)

en_loss, en_alpha, en_nnz = rolling_log_partial_enet_wfcv_loss_series(
    d_all, HAR, EXOG, WINDOW, START_DATE, DELTA)
loss_cols["log HAR + all-exog (partial EN)"] = en_loss

losses = pd.DataFrame(loss_cols).dropna()
assert losses.notna().all().all() and len(losses) > 0
print(f"MCS loss matrix: {losses.shape[0]} OOS days x {losses.shape[1]} models "
      f"({losses.index.min().date()} .. {losses.index.max().date()})")
print(f"partial-EN diagnostics: median alpha={np.median(en_alpha):.4g} "
      f"(min {en_alpha.min():.4g}, max {en_alpha.max():.4g}), "
      f"mean #non-zero exog coefs={en_nnz.mean():.2f} of {len(EXOG)}\n")

mcs = MCS(losses, size=0.05, reps=10000, block_size=None,
          method="max", bootstrap="stationary", seed=42)
mcs.compute()

pv = mcs.pvalues["Pvalue"]
mcs_results = (pd.DataFrame({
        "model": list(losses.columns),
        "mean_qlike": [losses[c].mean() for c in losses.columns],
        "pvalue": [float(pv[c]) for c in losses.columns],
        "in_mcs": [bool(pv[c] > 0.05) for c in losses.columns],
    })
    .sort_values("pvalue", ascending=False).reset_index(drop=True))
mcs_results.to_csv("partial_enet_comparison.csv", index=False)

pd.set_option("display.float_format", lambda v: f"{v:.6f}")
print(f"{int(mcs_results['in_mcs'].sum())} of {len(mcs_results)} models in the 5% MCS; "
      f"{int((~mcs_results['in_mcs']).sum())} excluded (p<=0.05).")
print(f"Lowest mean-QLIKE model: {losses.mean().idxmin()}  ({losses.mean().min():.6f})")
print("Saved -> partial_enet_comparison.csv\n")
mcs_results

MCS loss matrix: 2481 OOS days x 4 models (2016-07-11 .. 2026-05-28)
partial-EN diagnostics: median alpha=0.05109 (min 0.01212, max 10), mean #non-zero exog coefs=3.21 of 6



4 of 4 models in the 5% MCS; 0 excluded (p<=0.05).
Lowest mean-QLIKE model: log HAR + GVZ + macro  (0.027947)
Saved -> partial_enet_comparison.csv



,model,mean_qlike,pvalue,in_mcs
0,log HAR + GVZ + macro,0.027947,1.000000,True
1,log HAR + all-exog (OLS),0.027997,0.654100,True
2,log HAR + GVZ,0.028842,0.093400,True
3,log HAR + all-exog (partial EN),0.028958,0.093400,True
